In [1]:
import sys
!{sys.executable} -m pip install iiif_prezi3

In [2]:
import json
from SPARQLWrapper import SPARQLWrapper, JSON
from iiif_prezi3 import Canvas, Manifest, Annotation, AnnotationPage, ResourceItem

In [3]:
endpoint = "http://localhost:8089/blazegraph/sparql"
outputFile = "output/collectionObjects.json"

In [4]:
def sparqlResultToDict(results):
    rows = []
    for result in results["results"]["bindings"]:
        row = {}
        for key in list(result.keys()):
            row[key] = result[key]["value"]
        rows.append(row)
    return rows

In [5]:
sparql = SPARQLWrapper(endpoint)
sparql.setReturnFormat(JSON)

In [6]:
query = """
PREFIX la: <https://linked.art/ns/terms/>
PREFIX graph: <https://data.skkg.ch/graph/>
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?object ?label WHERE {
  ?object rdfs:label ?label ;
      crm:P138i_has_representation ?multimedia .
  ?multimedia la:digitally_shown_by  ?image .
}
"""
sparql.setQuery(query)
ret = sparqlResultToDict(sparql.queryAndConvert())

In [7]:
collectionManifest = {
  "@context": "http://iiif.io/api/presentation/3/context.json",
  "id": "https://skkg-iiif.swissartresearch.net/iiif/collection/objects",
  "type": "Collection",
  "label": { "en": [ "Collection of Objects with Images" ] },
  "items": []
}

In [8]:
for row in ret:
    item = {
        "id": row['object'].replace("https://data.skkg.ch/object", "https://skkg-iiif.swissartresearch.net/manifest/object"),
        "type": "Manifest",
        "label": {
            "none": [ row['label']]
        }
    }
    collectionManifest['items'].append(item)

In [9]:
with open(outputFile, 'w') as f:
    json.dump(collectionManifest, f, indent=4)